# React——前端开发新规则

## 框架是什么：管一类事情的一套规则

React 的核心规则可以先概括为：用组件描述界面，用数据驱动组件，用组件的组合构建页面。

## Vanilla：没有框架的原生写法

Vanilla 指不使用 React、Vue 等框架，只使用浏览器原生的 HTML、CSS 和 JavaScript。比如：

```javascript
const button = document.querySelector(".primary-button");
const output = document.querySelector("[data-output]");

button.addEventListener("click", () => {
  output.textContent = "已完成";
});
```

原生写法并没有错，小页面用它反而直接清楚。页面变大之后，需要自己管理 DOM 查询、事件绑定、状态同步和文件之间的依赖，代码容易分散。React 通过组件和声明式渲染来管理这些变化。

## 浏览器并不认识 React

浏览器真正认识的仍然只有 HTML、CSS 和 JavaScript。

Vite 会在开发和构建时把 JSX （React发明的语法糖），、模块导入等代码编译成浏览器可以执行的 JavaScript。React 负责运行时的组件和渲染逻辑，Vite 负责把开发者写的代码送到浏览器。

语法糖：没有增加新功能，只是换一套更好写、可读性更高的写法

## 把 React 装进项目

在已有的 Vite 项目根目录安装 React：

```bash
npm install react react-dom
npm install -D @vitejs/plugin-react
```

`react` 提供组件模型，`react-dom` 负责把 React 组件渲染到浏览器 DOM。`@vitejs/plugin-react` 插件让 Vite 识别 JSX，并提供开发期的 React 支持。

项目根目录下新建文件 `vite.config.js`（把插件挂进 vite）这样配置：

```javascript
import { defineConfig } from "vite";
import react from "@vitejs/plugin-react";

export default defineConfig({
  plugins: [react()],
});
```

## 这节课的改造路线

查看本地电脑的家目录下拉取的zero-to-tech-4-3

逐步迁移：

1. 先把结果区的一张卡片变成 React 组件。
2. 再把文字实验室页面交给 React。
3. 把个人主页也抽成组件，观察复用的价值。
4. 最后把两个页面收进 `App`，换成新的 React 入口。

每一步都能运行，出现问题时容易定位，也能清楚看到框架带来的变化。

## 第一拍：先把结果区卡片做成 React 组件

### 新建文件夹 `src/components/`

将`ResultCard.jsx`组件文件复制进 `src/components/`。一个组件，就是把“数据、结构、样式、行为”这一整套，封装成一个能独立拎走的 UI 单元（界面上独立、可以拆分出来的一块视图单元）。

### 新建入口文件 `src/result.jsx`

React 组件需要一个入口才能挂到 HTML：

```jsx
import { createRoot } from "react-dom/client";
import ResultCard from "./components/ResultCard.jsx";

const root = createRoot(document.getElementById("result-root"));
root.render(<ResultCard />);
```

### 删掉 `text-lab.html` 里的结果区 article class="panel panel-half lab-panel result-panel card" 换为

```html
<div id="result-root" class="panel-half"></div>
```

### 留出挂载点，在页面底部加上第二行

```html
<script type="module" src="js/main.js"></script>
<script type="module" src="/src/result.jsx"></script>
```

`createRoot` 把 React 和页面中的某个 DOM 节点连接起来。运行 `npm run dev` 后，结果卡片就由 React 渲染。

## 读懂 `ResultCard`：一个完整的 UI 组件

一个组件通常包括四部分：

- 数据：要显示的标题、数值、标签等。
- 结构：组件返回的 JSX。
- 样式接口：组件使用的 class 名称。
- 行为：按钮事件、输入响应或动画触发。

```jsx
function ResultCard({ title, value, description }) {
  return (
    <article className="result-card">
      <h3>{title}</h3>
      <strong>{value}</strong>
      <p>{description}</p>
    </article>
  );
}

export default ResultCard;
```

大写开头的 `ResultCard` 是组件；小写开头的 `article`、`h3`、`strong` 和 `p` 是 HTML 标签。JSX 中用 `{}` 插入 JavaScript 表达式。

## 第二拍：把整个文字实验室页面交给 React

### 拷入页面相关组件

这一页天然可以拆成几块：

- 顶部导航：Nav
- 大标题和副标题：PageHeading
- 输入卡片：InputCard
- 结果卡片：ResultCard
- 外层动画和网格容器：AnimatedCardGrid

AnimatedCardGrid 这类组件自己不一定展示具体内容，它更像一个容器：把别的组件包进去，统一提供布局和动画

再用一个大组件`TextLabPage.jsx`把他们拼成整页

把 `css` 文件夹拷到 `src/css/` : 样式也会改由 React 入口文件 import 进来。

根目录下的`css`还在被个人主页 index.html 此使用

### 换掉临时入口

删掉`src/result.jsx`,新建`src/textlab.jsx`入口文件，复制进下面的内容

```jsx
import { createRoot } from "react-dom/client";
import TextLabPage from "./components/TextLabPage.jsx";

import "./css/reset.css";
import "./css/variables.css";
import "./css/layout.css";
import "./css/hero.css";
import "./css/nav.css";
import "./css/cards.css";
import "./css/lab.css";
import "./css/responsive.css";

createRoot(document.getElementById("root")).render(
  <div className="app-shell">
    <div className="page-shell">
      <main className="page-content">
        <TextLabPage current="textlab" onNavigate={() => {}} />
      </main>
    </div>
  </div>,
);
```

### 把 text-lab.html 的 <body> 整块内容替换：

```html
<body>
  <div id="root"></div>
  <script type="module" src="/src/textlab.jsx"></script>
</body>
```

<head> 里那些 CSS <link> 也删掉

## 看懂 `TextLabPage`：组件可以嵌套组件

```jsx
function TextLabPage() {
  return (
    <>
      <Nav />
      <PageHeading title="文字实验室" subtitle="拼音和情绪，挖掘中文里的细节" />
      <InputCard />
      <ResultCard />
      <AnimatedCardGrid />
    </>
  );
}
```

组件挂进 HTML 需要入口文件，组件放进组件只需要像写标签一样使用它。这样页面形成一棵组件树。

- **DOM**：浏览器内存，HTML 标签的树（真实页面节点）
- **组件树**：React 代码层面，你的组件函数互相嵌套形成的树。

### props：给组件传参数

```jsx
<PageHeading title="文字实验室" subtitle="拼音和情绪，挖掘中文里的细节" />
```

里面的 title 和 subtitle，就是传给组件的参数，React 里通常叫 props

## 第三拍：把个人主页也抽成组件

有些小组件可以复用，修改 props 就行。将`HomePage.jsx`复制到项目中。

个人主页下面那两张卡片没有被单独拆成组件，而是直接写在 HomePage 里。

组件不是拆得越细越好，拆分的理由通常是：需要复用、或者拆开后页面结构更容易阅读。

## 把两个页面收进 `App`

两个页面都已经是组件，就可以由一个顶层组件**APP**统一组织，从 Demo 里铐入`src/App.jsx`

## 第四拍：全新的 `index.html` 和 `main.jsx`

### 清理旧文件

两个 HTML 文件的内容已经交给 React 。

删掉：

- 旧 index.html
- text-lab.html
- 第二拍临时用的 src/textlab.jsx
- 旧的 js/ 文件夹
- 旧的 css/ 文件夹

### 拷入 `src/main.jsx`

`src/main.jsx`入口文件就是整个 React 项目的总入口。


### 项目根目录下新建全新的 `index.html`

```html
<!doctype html>
<html lang="zh-CN">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>Zero to Tech</title>
  </head>
  <body>
    <div id="root"></div>
    <script type="module" src="/src/main.jsx"></script>
  </body>
</html>
```

新的 `index.html` 只需要做一件事：提供挂载点，并引入 main.jsx。

### 运行和构建

```bash
npm run dev        # 查看新页面
npm run build      # 重新构建
```

## 最终的 React 项目骨架

```text
index.html
└── src/main.jsx
    └── App.jsx
        ├── HomePage.jsx
        │   ├── Nav.jsx
        │   └── PageHeading.jsx
        └── TextLabPage.jsx
            ├── Nav.jsx
            ├── PageHeading.jsx
            ├── InputCard.jsx
            ├── ResultCard.jsx
            └── AnimatedCardGrid.jsx
```

- index.html：空壳，只提供挂载点。
- main.jsx：入口，把 App 挂进去。
- App.jsx：总管，决定当前显示哪个页面。
- HomePage / TextLabPage：页面组件。
- 更小的组件：页面里的零件。

## 真正从 0 新建 React 项目

如果不改造旧项目，也可以让 Vite 直接生成 React 模板：

```bash
npm create vite
```

交互过程中选择项目名称、框架 `React` 和 JavaScript 或 TypeScript 变体，然后进入项目目录：

```bash
cd 项目目录
npm install
npm run dev
```

模板会预先准备 `index.html`、`src/main.jsx`、`src/App.jsx`、`package.json` 和 Vite 配置。